In [1]:
!pip install -q -U pandas rank_bm25 tqdm sentence-transformers pyvi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dopamine-rl 4.1.2 requires 

# Data for Finetune Embedding Model

In [3]:
import pandas as pd
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from pyvi import ViTokenizer
import os
import pickle
import re

START_IDX = 480001
END_IDX = 500000

def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

df = pd.read_parquet("hf://datasets/phamson02/large-vi-legal-queries/large_vi_legal_queries.parquet")
df_emb = df[['passage_id', 'query', 'context']].copy()
df_emb.columns = ['passage_id', 'query', 'passage']

print("Đang clean text...")
df_emb['query'] = df_emb['query'].apply(clean_text)
df_emb['passage'] = df_emb['passage'].apply(clean_text)

OUTPUT_FILE = f"viet_legal_triplets_{START_IDX}_{END_IDX}.csv"
CORPUS_FILE = "/kaggle/input/datasets/ngpham06/pkl-corpus/corpus_tokenized.pkl"

tqdm.pandas()

corpus_raw = df_emb['passage'].unique().tolist()

if os.path.exists(CORPUS_FILE):
    print("Tìm thấy corpus đã tokenize, đang load...")
    with open(CORPUS_FILE, 'rb') as f:
        tokenized_corpus = pickle.load(f)
    print("Load corpus xong!")
else:
    print("Chưa có corpus tokenize, đang tokenize lần đầu...")
    tokenized_corpus = []
    for doc in tqdm(corpus_raw, desc="Tokenize corpus"):
        tokenized_corpus.append(ViTokenizer.tokenize(clean_text(doc)).split())
    with open(CORPUS_FILE, 'wb') as f:
        pickle.dump(tokenized_corpus, f)
    print("Đã lưu corpus tokenize!")

corpus_tokenized_str = [" ".join(tokens) for tokens in tokenized_corpus]

print("Đang khởi tạo BM25...")
bm25 = BM25Okapi(tokenized_corpus)
print("Khởi tạo BM25 xong!")



Đang clean text...
Tìm thấy corpus đã tokenize, đang load...
Load corpus xong!
Đang khởi tạo BM25...
Khởi tạo BM25 xong!


In [4]:
def get_hard_negative(query, positive_text):
    tokenized_query = str(query).split()
    top_n = bm25.get_top_n(tokenized_query, corpus_tokenized_str, n=10)
    for doc in top_n:
        if clean_text(doc) != clean_text(positive_text):
            return doc
    return None

df_slice = df_emb.iloc[START_IDX:END_IDX].copy()
print(f"Xử lý từ dòng {START_IDX} đến {END_IDX}: {len(df_slice)} dòng")

print("Đang tokenize query...")
df_slice['query'] = df_slice['query'].progress_apply(
    lambda x: ViTokenizer.tokenize(str(x))
)

print("Đang tokenize passage...")
df_slice['passage'] = df_slice['passage'].progress_apply(
    lambda x: ViTokenizer.tokenize(str(x))
)

print("Đang đào Hard Negatives...")
df_slice['negative'] = df_slice.progress_apply(
    lambda x: get_hard_negative(x['query'], x['passage']), axis=1
)

df_slice = df_slice.rename(columns={'query': 'anchor', 'passage': 'positive'})
df_slice = df_slice.dropna(subset=['negative'])
df_output = df_slice[['passage_id', 'anchor', 'positive', 'negative']]
df_output.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f"Hoàn thành! Đã lưu {len(df_output)} dòng vào {OUTPUT_FILE}")

Xử lý từ dòng 480001 đến 500000: 19999 dòng
Đang tokenize query...


100%|██████████| 19999/19999 [00:09<00:00, 2009.32it/s]


Đang tokenize passage...


100%|██████████| 19999/19999 [01:24<00:00, 237.59it/s]


Đang đào Hard Negatives...


100%|██████████| 19999/19999 [6:26:46<00:00,  1.16s/it]


Hoàn thành! Đã lưu 19999 dòng vào viet_legal_triplets_480001_500000.csv
